# Unsupervised Learning: Step-by-Step

By the end of this notebook, you will understand:

- What unsupervised learning is and how it differs from supervised learning
- The curse of dimensionality and why dimensionality reduction is important
- How to perform clustering with K-Means, Hierarchical, and DBSCAN
- How to reduce dimensions using PCA, t-SNE, and UMAP
- How to combine clustering and dimensionality reduction
- How to visualize and interpret results
- How to experiment interactively with parameters


Text Cell 2: Step 0 — Install and Import Libraries

In [ ]:
# Step 0: Install and import required libraries

We will use:
- `numpy` and `pandas` for data handling
- `matplotlib` and `seaborn` for visualization
- `scikit-learn` for clustering and PCA/t-SNE
- `umap-learn` for UMAP dimensionality reduction
- `ipywidgets` for interactive exploration

In [ ]:
!pip install umap-learn ipywidgets --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
import umap
import ipywidgets as widgets
from ipywidgets import interact

sns.set(style="whitegrid")
%matplotlib inline


 Step 1 — Create a Synthetic Dataset

In [ ]:
We will simulate a **stellar dataset** with 500 stars. Each star has features:

- Temperature
- Metallicity
- Luminosity
- Radius

This synthetic dataset allows us to explore clustering and dimensionality reduction interactively.


In [ ]:
np.random.seed(42)
n_stars = 500

temperature = np.random.normal(5500, 500, n_stars)
metallicity = np.random.normal(0, 0.2, n_stars)
luminosity = np.random.normal(1.0, 0.3, n_stars)
radius = np.random.normal(1.0, 0.2, n_stars)

stars = pd.DataFrame({
    "Temperature": temperature,
    "Metallicity": metallicity,
    "Luminosity": luminosity,
    "Radius": radius
})

stars.head()


Step 2 — Standardize the Data

In [ ]:
features = ["Temperature", "Metallicity", "Luminosity", "Radius"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(stars[features])


Step 3 — Interactive Clustering


Now we will cluster the stars using **K-Means**, **Hierarchical Clustering**, or **DBSCAN**.  

Use the interactive widgets below to choose:
- Clustering algorithm
- Number of clusters (for K-Means and Hierarchical)
- DBSCAN parameters (`eps` and `min_samples`)

In [ ]:
def interactive_clustering(method='K-Means', n_clusters=3, eps=0.8, min_samples=5):
    plt.figure(figsize=(8,6))

    if method == 'K-Means':
        model = KMeans(n_clusters=n_clusters, random_state=42)
        labels = model.fit_predict(X_scaled)
    elif method == 'Hierarchical':
        model = AgglomerativeClustering(n_clusters=n_clusters)
        labels = model.fit_predict(X_scaled)
    elif method == 'DBSCAN':
        model = DBSCAN(eps=eps, min_samples=min_samples)
        labels = model.fit_predict(X_scaled)
    else:
        raise ValueError("Invalid method")

    plt.scatter(stars["Temperature"], stars["Luminosity"], c=labels, cmap='Set2', s=60)
    plt.xlabel("Temperature")
    plt.ylabel("Luminosity")
    plt.title(f"{method} Clustering")
    plt.show()

    # Compute silhouette score only if clusters >1
    if len(set(labels)) > 1 and -1 not in set(labels):
        score = silhouette_score(X_scaled, labels)
        print(f"Silhouette Score: {score:.2f}")
    elif method != 'DBSCAN':
        score = silhouette_score(X_scaled, labels)
        print(f"Silhouette Score: {score:.2f}")
    else:
        print("Silhouette Score not computed for DBSCAN with noise points (-1 labels).")

interact(interactive_clustering,
         method=['K-Means','Hierarchical','DBSCAN'],
         n_clusters=widgets.IntSlider(value=3, min=2, max=10, step=1),
         eps=widgets.FloatSlider(value=0.8, min=0.1, max=3.0, step=0.1),
         min_samples=widgets.IntSlider(value=5, min=1, max=20, step=1));


Step 4 — Interactive Dimensionality Reduction

We can now reduce the dimensions of our dataset to **2D** for visualization:

- **PCA**: Linear reduction, captures the largest variance
- **t-SNE**: Nonlinear reduction, preserves local clusters
- **UMAP**: Nonlinear, preserves local and global structure

Use the interactive widget to choose the technique and color points by a feature.


In [ ]:
def interactive_dim_reduction(method='PCA', color_feature='Temperature'):
    plt.figure(figsize=(8,6))

    X = X_scaled

    if method == 'PCA':
        reducer = PCA(n_components=2)
        X_red = reducer.fit_transform(X)
        explained_var = reducer.explained_variance_ratio_
        print(f"Explained Variance: PC1={explained_var[0]:.2f}, PC2={explained_var[1]:.2f}")
    elif method == 't-SNE':
        reducer = TSNE(n_components=2, random_state=42)
        X_red = reducer.fit_transform(X)
    elif method == 'UMAP':
        reducer = umap.UMAP(n_components=2, random_state=42)
        X_red = reducer.fit_transform(X)

    plt.scatter(X_red[:,0], X_red[:,1], c=stars[color_feature], cmap='viridis', s=50)
    plt.colorbar(label=color_feature)
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.title(f"{method} Projection of Stellar Data")
    plt.show()

interact(interactive_dim_reduction,
         method=['PCA','t-SNE','UMAP'],
         color_feature=['Temperature','Luminosity','Metallicity','Radius']);


Step 5 — Combine Clustering with Dimensionality Reduction

We can now **cluster in the reduced space** (e.g., after PCA) to see if dimensionality reduction helps reveal clusters more clearly.


In [ ]:
def clustering_on_reduced(dr_method='PCA', cluster_method='K-Means', n_clusters=3):
    # Dimensionality reduction
    if dr_method == 'PCA':
        X_red = PCA(n_components=2).fit_transform(X_scaled)
    elif dr_method == 't-SNE':
        X_red = TSNE(n_components=2, random_state=42).fit_transform(X_scaled)
    elif dr_method == 'UMAP':
        X_red = umap.UMAP(n_components=2, random_state=42).fit_transform(X_scaled)

    # Clustering
    if cluster_method == 'K-Means':
        labels = KMeans(n_clusters=n_clusters, random_state=42).fit_predict(X_red)
    elif cluster_method == 'Hierarchical':
        labels = AgglomerativeClustering(n_clusters=n_clusters).fit_predict(X_red)

    plt.figure(figsize=(8,6))
    plt.scatter(X_red[:,0], X_red[:,1], c=labels, cmap='Set2', s=60)
    plt.xlabel(f"{dr_method} Component 1")
    plt.ylabel(f"{dr_method} Component 2")
    plt.title(f"{cluster_method} on {dr_method}-Reduced Data")
    plt.show()

interact(clustering_on_reduced,
         dr_method=['PCA','t-SNE','UMAP'],
         cluster_method=['K-Means','Hierarchical'],
         n_clusters=widgets.IntSlider(value=3, min=2, max=10, step=1));


Step 6 — Exercises

### Exercises

1. Explore different numbers of clusters in K-Means and Hierarchical Clustering.
2. Try DBSCAN with different `eps` and `min_samples`.
3. Compare PCA, t-SNE, and UMAP projections. Which shows clusters most clearly?
4. Color by different features and observe patterns.
5. Combine clustering with dimensionality reduction and see how it improves cluster separation.
6. Think of a dataset from your field and consider how you would apply these techniques.
